In [ ]:
# ============================================================
# SETUP — Drive already mounted, skip remount
# ============================================================
import os, sys
import numpy as np, pandas as pd, torch

PROJECT_ROOT = os.path.join(DATA_DIR, "..")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
SRC_DIR       = os.path.join(PROJECT_ROOT, "src")
sys.path.insert(0, SRC_DIR)
from preprocessing import FINAL_FEATURES_FD001
print(f"Sensors: {FINAL_FEATURES_FD001}")

# --- Load data ---
train     = pd.read_parquet(os.path.join(PROCESSED_DIR, "train_FD001_processed.parquet"))
test_full = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_processed.parquet"))
test_last = pd.read_parquet(os.path.join(PROCESSED_DIR, "test_FD001_last_cycle.parquet"))
print(f"train {train.shape}, test_full {test_full.shape}, test_last {test_last.shape}")

# --- Load Chronos-2 pipeline ---
from chronos import Chronos2Pipeline
pipe2 = Chronos2Pipeline.from_pretrained("amazon/chronos-2")
print("✓ Chronos-2 loaded")

# --- pool_embedding helper ---
def pool_embedding(e):
    e = e.detach().float().cpu().numpy() if hasattr(e, "detach") else np.asarray(e)
    return e.reshape(-1, e.shape[-1]).mean(axis=0)

print("✓ Setup complete — now run the Chronos full-history cell.")

In [ ]:
# ============================================================
# 10B_Chronos_Full_History.ipynb  (full history, checkpointed)
# ============================================================
# Fair comparison: give Chronos-2 each engine's FULL history.
# Needs from NB06 in memory: pipe2, pool_embedding, train,
# test_full, test_last, FINAL_FEATURES_FD001.
# Embeddings are SAVED incrementally -> safe if the session drops.
# ============================================================
import numpy as np, pandas as pd, os, time, torch
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

RESULTS_DIR = os.path.join(DATA_DIR, "..", "results")
EMB_DIR     = os.path.join(DATA_DIR, "..", "embeddings_full")
os.makedirs(EMB_DIR, exist_ok=True)
Y_MAX = 125.0
FEATS = FINAL_FEATURES_FD001

def nasa_score(y_t, y_p):
    d = y_p - y_t
    return float(np.sum(np.where(d < 0, np.exp(-d/13.0)-1.0, np.exp(d/10.0)-1.0)))

# ---- Embed ONE full-history window -> pooled vector ----
def embed_one(w):
    x = torch.tensor(w.T[None, :, :], dtype=torch.float32)  # (1, 13, W)
    out = pipe2.embed(x)
    emb = out[0] if isinstance(out, tuple) else out
    if isinstance(emb, list):
        return pool_embedding(emb[0])
    e = emb.detach().float().cpu().numpy()
    return e.reshape(e.shape[0], -1, e.shape[-1]).mean(axis=1)[0]

# ============================================================
# STEP 1 — TEST embeddings (only 100, fast). Full observed history.
# ============================================================
test_emb_path = os.path.join(EMB_DIR, "test_full_emb.npy")
if os.path.exists(test_emb_path):
    X_te = np.load(test_emb_path)
    print(f"Loaded cached TEST embeddings: {X_te.shape}")
else:
    print("Embedding TEST engines (full history)...")
    units_test = test_last["unit"].values
    t0 = time.time(); embs = []
    for i, unit in enumerate(units_test):
        g = test_full[test_full["unit"]==unit].sort_values("cycle")
        embs.append(embed_one(g[FEATS].values.astype(np.float32)))
        if (i+1) % 20 == 0:
            print(f"  {i+1}/100  ({time.time()-t0:.0f}s)")
    X_te = np.stack(embs).astype(np.float32)
    np.save(test_emb_path, X_te)
    print(f"Saved TEST embeddings: {X_te.shape} in {time.time()-t0:.0f}s")

y_true = test_last["RUL_true"].values

# ============================================================
# STEP 2 — TRAIN embeddings (full history for every cycle).
# Checkpointed every 500. Prints ETA so you can decide.
# ============================================================
# Build the list of (unit, end_cycle) samples
samples = []
for unit, g in train.groupby("unit"):
    g = g.sort_values("cycle")
    arr = g[FEATS].values.astype(np.float32)
    pw  = g["RUL_piecewise125"].values.astype(np.float32)
    for end in range(len(g)):
        samples.append((arr[0:end+1], pw[end]))
N = len(samples)
print(f"\nTotal TRAIN windows to embed: {N}")

train_emb_path = os.path.join(EMB_DIR, "train_full_emb.npy")
train_lab_path = os.path.join(EMB_DIR, "train_full_lab.npy")
ckpt_path      = os.path.join(EMB_DIR, "train_progress.npy")

# Resume if partial progress exists
if os.path.exists(train_emb_path) and os.path.exists(ckpt_path):
    X_tr = list(np.load(train_emb_path, allow_pickle=True))
    y_tr = list(np.load(train_lab_path, allow_pickle=True))
    start = int(np.load(ckpt_path))
    print(f"Resuming from sample {start}/{N}")
else:
    X_tr, y_tr, start = [], [], 0

t0 = time.time()
for i in range(start, N):
    w, lab = samples[i]
    X_tr.append(embed_one(w)); y_tr.append(lab)
    if (i+1) % 500 == 0:
        np.save(train_emb_path, np.array(X_tr, dtype=np.float32))
        np.save(train_lab_path, np.array(y_tr, dtype=np.float32))
        np.save(ckpt_path, i+1)
        elapsed = time.time()-t0
        rate = (i+1-start)/elapsed
        eta = (N-(i+1))/rate/60
        print(f"  {i+1}/{N}  | {rate:.1f} win/s | ETA {eta:.0f} min | saved")

# final save
X_tr = np.array(X_tr, dtype=np.float32); y_tr = np.array(y_tr, dtype=np.float32)
np.save(train_emb_path, X_tr); np.save(train_lab_path, y_tr); np.save(ckpt_path, N)
print(f"\nDONE embedding train: {X_tr.shape} in {(time.time()-t0)/60:.1f} min")

# ============================================================
# STEP 3 — Train head (Ridge) + evaluate
# ============================================================
scaler = StandardScaler().fit(X_tr)
Xtr_s, Xte_s = scaler.transform(X_tr), scaler.transform(X_te)
head = Ridge(alpha=10.0).fit(Xtr_s, y_tr)
y_pred = np.clip(head.predict(Xte_s), 0, 125)

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
ns   = nasa_score(y_true, y_pred)
print("\n" + "="*60)
print("CHRONOS-2 (FULL HISTORY) — RESULT")
print("="*60)
print(f"  MAE {mae:.2f} | RMSE {rmse:.2f} | NASA {ns:.2f}")

pd.DataFrame({"unit": test_last["unit"].values, "RUL_true": y_true,
              "pred_Chronos_full": y_pred}).to_csv(
    os.path.join(RESULTS_DIR, "fd001_chronos_full_history.csv"), index=False)

# ============================================================
# FINAL FAIR TABLE
# ============================================================
print("\n" + "="*70)
print("FAIR COMPARISON — every model sees each engine's FULL history")
print("="*70)
print(f"{'Model':<18}{'Input form':<26}{'MAE':>7}{'RMSE':>7}{'NASA':>9}")
print("-"*70)
print(f"{'LR':<18}{'65 features':<26}{14.99:>7.2f}{17.98:>7.2f}{514.27:>9.2f}")
print(f"{'XGBoost':<18}{'65 features':<26}{8.57:>7.2f}{12.01:>7.2f}{242.67:>9.2f}")
print(f"{'LSTM (full)':<18}{'all cycles':<26}{10.16:>7.2f}{13.79:>7.2f}{313.54:>9.2f}")
print(f"{'Chronos (full)':<18}{'all cycles':<26}{mae:>7.2f}{rmse:>7.2f}{ns:>9.2f}")
print("-"*70)